# Fig 6a, Supp Fig 6a–b: NicheNet circos plots of the selected communication axes

Produces:
- **Fig 6a**: circos plot of the selected NicheNet axes in border zone 1 (BZ1), TIL-high
  (orange) and TIL-low (blue) axes; senders and ligands on one side, receivers and target
  genes on the other; cord width ~ ligand activity, colour ~ rank score.
- **Supp Fig 6a**: the same plot for TIL+ spots.
- **Supp Fig 6b**: the same plot for border zone 2 (BZ2).
- The axes named in the Results text, printed from the plotted tables.

Origin: the circos cells of `pipelines/NicheNet_v3.ipynb` (Spencer), cells 80 (TIL+),
116 (BZ1) and 152 (BZ2), identical apart from the zone passed to R.

Inputs are the frozen `axes_selected.csv` of each zone in `config.NICHENET_DIR/{TILplus,BZ1,BZ2}/`,
written by the axis-selection cells of `pipelines/NicheNet_v3.ipynb`. The TIL+ file predates
the current TIL+ axis catalog (6 of its 54 axis IDs are not in `TILplus/axes_catalog.csv`,
printed below), so it cannot be rebuilt from the notebook state; it is used as frozen.

Changes from the source cells:
- The R code of the cell is kept as written, wrapped in one function,
  `gbmcollaboration.paper.nichenet.circos_zone(axes_csv, out_png)` (R circlize via rpy2,
  in place of the `%%R` cell magic), called once per zone instead of three copies of the cell.
  Only the head changed: the input CSV and output PNG are arguments in place of the
  `ZONE` → folder path lines.
- Figures go to `OUT` (the notebook wrote them next to `axes_selected.csv`), named by panel;
  the plotted axes (with cord widths, colours and positions) are saved as `<panel>.csv`;
  PNG, PDF and CSV are also copied to the panels folder (`Fig6a`, `SuppFig6a`, `SuppFig6b`).
- New: comparison with the published PNGs, the TIL+ catalog check and the printout of the
  axes named in the Results.
- R circlize, dplyr and readr were added to the `gbm_main` environment (conda-forge).

In [ ]:
import filecmp
import shutil
import time

import numpy as np
import pandas as pd
from PIL import Image

from gbmcollaboration.paper import config, output
from gbmcollaboration.paper.nichenet import circos_zone

In [ ]:
# Parameters
OUT = output.setup("20_NicheNet_circos")
NICHENET_DIR = config.NICHENET_DIR

# panel -> zone folder, in manuscript order
PANELS = {"fig6a_circos_BZ1": "BZ1", "suppfig6a_circos_TILplus": "TILplus", "suppfig6b_circos_BZ2": "BZ2"}
# manuscript panel name of each figure (also written to output.PANELS_DIR)
PAPER_PANEL = {"fig6a_circos_BZ1": "Fig 6a", "suppfig6a_circos_TILplus": "Supp Fig 6a", "suppfig6b_circos_BZ2": "Supp Fig 6b"}
# Published circos file of the notebook run, in each zone folder
PUBLISHED_PNG = "circos_selected_axes_senders_receivers_only.png"

# Ligands of the axes named in the Results text
TEXT_LIGANDS = {
    "suppfig6a_circos_TILplus": ["TGFB3", "FGF2", "HGF", "TGFB1", "TGFB2", "APOD", "IL15"],
    "fig6a_circos_BZ1": ["TGFB3", "CCN2", "IGF1", "IL10"],
    "suppfig6b_circos_BZ2": ["TGFB3", "CCN2", "BMP2", "TGFB1", "PDGFA", "IL1B"],
}

In [ ]:
def save_panel(panel: str) -> None:
    """Plotted axes to OUT/<panel>.csv; the R-written PNG/PDF and the CSV also to output.PANELS_DIR."""
    output.save_table(OUT, panel, AXES[panel], panel=PAPER_PANEL[panel], index=False)
    for ext in ("png", "pdf"):
        shutil.copyfile(OUT / f"{panel}.{ext}", output.PANELS_DIR / f"{output.panel_name(PAPER_PANEL[panel])}.{ext}")


def compare_with_published(panel: str, zone_dir: str) -> pd.Series:
    """Byte and pixel comparison of the new PNG with the notebook's PNG of the same zone."""
    new_png, old_png = OUT / f"{panel}.png", NICHENET_DIR / zone_dir / PUBLISHED_PNG
    new = np.asarray(Image.open(new_png).convert("RGB")).astype(int)
    old = np.asarray(Image.open(old_png).convert("RGB")).astype(int)
    res = {"byte-identical": filecmp.cmp(new_png, old_png, shallow=False), "same size": new.shape == old.shape}
    if new.shape == old.shape:
        diff = np.abs(new - old).max(axis=2)
        res |= {"identical pixels": bool((diff == 0).all()),
                "pixels differing (%)": round(100 * (diff > 0).mean(), 3)}
    return pd.Series(res)


AXES = {}
CHECKS = {}
t0 = time.time()

## Fig 6a: BZ1

Selected axes of border zone 1, TIL-high against TIL-low samples.

In [ ]:
AXES["fig6a_circos_BZ1"] = circos_zone(NICHENET_DIR / "BZ1" / "axes_selected.csv", OUT / "fig6a_circos_BZ1.png")
save_panel("fig6a_circos_BZ1")
CHECKS["fig6a_circos_BZ1"] = compare_with_published("fig6a_circos_BZ1", "BZ1")
print(CHECKS["fig6a_circos_BZ1"].to_string())

## Supp Fig 6a: TIL+

Selected axes of TIL+ spots. The axis file predates the current TIL+ catalog (see top).

In [ ]:
AXES["suppfig6a_circos_TILplus"] = circos_zone(NICHENET_DIR / "TILplus" / "axes_selected.csv",
                                               OUT / "suppfig6a_circos_TILplus.png")
save_panel("suppfig6a_circos_TILplus")
catalog = pd.read_csv(NICHENET_DIR / "TILplus" / "axes_catalog.csv", usecols=["axis_id"])
missing = sorted(set(AXES["suppfig6a_circos_TILplus"]["axis_id"]) - set(catalog["axis_id"].astype(str)))
print(f"TIL+ axis IDs not in the current TILplus/axes_catalog.csv: {len(missing)} {missing}")
CHECKS["suppfig6a_circos_TILplus"] = compare_with_published("suppfig6a_circos_TILplus", "TILplus")
print(CHECKS["suppfig6a_circos_TILplus"].to_string())

## Supp Fig 6b: BZ2

Selected axes of border zone 2.

In [ ]:
AXES["suppfig6b_circos_BZ2"] = circos_zone(NICHENET_DIR / "BZ2" / "axes_selected.csv", OUT / "suppfig6b_circos_BZ2.png")
save_panel("suppfig6b_circos_BZ2")
CHECKS["suppfig6b_circos_BZ2"] = compare_with_published("suppfig6b_circos_BZ2", "BZ2")
print(CHECKS["suppfig6b_circos_BZ2"].to_string())
print(f"\nThree circos plots in {time.time() - t0:.0f} s")

## Axes named in the Results

The sender → ligand → receiver → target axes behind the sentences of the Results, read from
the plotted tables (one line per ligand and direction).

In [ ]:
for panel, ligands in TEXT_LIGANDS.items():
    a = AXES[panel]
    a = a[a["ligand"].isin(ligands)]
    summary = (a.groupby(["direction", "ligand"])
               .apply(lambda d: pd.Series({
                   "senders": ", ".join(sorted(d["sender"].unique())),
                   "receiver: target": ", ".join(sorted(set(d["receiver"] + ": " + d["target"])))}), include_groups=False)
               .reset_index())
    print(f"== {panel}")
    print(summary.to_string(index=False))

In [ ]:
print(output.versions(("numpy", "pandas", "pillow", "rpy2"), r_packages=("circlize", "dplyr", "readr")).to_string(index=False))